# PySpark String & Date Functions

**Topics:** Conditional logic, string operations, date conversions, handling duplicates and nulls

In [7]:
# Initialize Spark Session
import findspark

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("strings-dates").master("local[*]").getOrCreate()

In [8]:
# Create sample employee data with data quality issues
# Issues: duplicates, missing values, inconsistent gender formats (Male/Female/M/F)

emp_schema = """
    emp_id STRING, 
    dept_id STRING, 
    name STRING, 
    age STRING, 
    gender STRING, 
    hire_date STRING,
    salary STRING
"""

emp_data = [
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],
    ["003", "101", "Priya S", "24", "Female", "2024-03-10", "62000"],
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],  # duplicate
    ["004", "103", "Meena R", "", "Female", "2022-11-20", ""],  # missing age & salary
    ["005", "102", "Saravanan", "35", "Male", "", "95000"],
    ["006", "104", "Karthik P", "29", "Male", "2024-09-05", "72000"],
    ["007", "", "Deepika Menon", "26", "Female", "2023-02-28", "68000"],  # missing dept
    ["008", "101", "Mohan Raj", "42", "Male", "2020-05-12", "120000"],
    ["009", "103", "Anjali", "23", "F", "2024-07-19", "58000"],
    ["010", "102", "Ramesh Kumar", "31", "M", "2021-10-01", "85000"],
    ["011", "105", "Swathi", "", "Female", "2025-02-10", "52000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],  # duplicate
    ["012", "101", "Vikram Singh", "38", "Male", "2019-08-25", "105000"],
    ["013", "104", "Preethi K", "27", "Female", "", "64000"],
    ["014", "103", "Naveen", "", "Male", "2024-01-15", "null"],  # explicit null
    ["015", "102", "Lavanya", "24", "Female", "2024-04-30", "61000"],
    ["016", "", "Sundar", "45", "M", "2018-03-05", "92000"],
    ["017", "101", "Kavya Sri", "22", "F", "2025-03-01", "48000"],
    ["018", "106", "Abdul Rahman", "33", "Male", "2022-12-12", "88000"],
]

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)
emp.show()

+------+-------+-------------+---+------+----------+------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|
+------+-------+-------------+---+------+----------+------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|
|   004|    103|      Meena R|   |Female|2022-11-20|      |
|   005|    102|    Saravanan| 35|  Male|          | 95000|
|   006|    104|    Karthik P| 29|  Male|2024-09-05| 72000|
|   007|       |Deepika Menon| 26|Female|2023-02-28| 68000|
|   008|    101|    Mohan Raj| 42|  Male|2020-05-12|120000|
|   009|    103|       Anjali| 23|     F|2024-07-19| 58000|
|   010|    102| Ramesh Kumar| 31|     M|2021-10-01| 85000|
|   011|    105|       Swathi|   |Female|2025-02-10| 52000|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|
|   012|    101| Vikram Singh| 38|  Male

In [9]:
# Conditional transformation: Standardize gender values using when/otherwise
# Converts "Male" → "M" and "Female" → "F"

from pyspark.sql.functions import col, when

emp_gender_fixed = emp.withColumn(
    "new_gender",
    when(col("gender") == "Male", "M")
    .when(col("gender") == "Female", "F")
    .otherwise(None),
)

In [10]:
# Display result - gender is now standardized
emp_gender_fixed.show(5)

+------+-------+----------+---+------+----------+------+----------+
|emp_id|dept_id|      name|age|gender| hire_date|salary|new_gender|
+------+-------+----------+---+------+----------+------+----------+
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|
|   002|    102|Arun Kumar| 28|  Male|2023-06-15| 78000|         M|
|   003|    101|   Priya S| 24|Female|2024-03-10| 62000|         F|
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|
|   004|    103|   Meena R|   |Female|2022-11-20|      |         F|
+------+-------+----------+---+------+----------+------+----------+
only showing top 5 rows


In [11]:
# String replacement: Replace "Vishnu" with "Dharshan" using regexp_replace

from pyspark.sql.functions import regexp_replace

emp_name_changed = emp_gender_fixed.withColumn(
    "new_name", regexp_replace(col("name"), "Vishnu", "Dharshan")
)

In [12]:
# Verify name replacement
emp_name_changed.show(5)

+------+-------+----------+---+------+----------+------+----------+----------+
|emp_id|dept_id|      name|age|gender| hire_date|salary|new_gender|  new_name|
+------+-------+----------+---+------+----------+------+----------+----------+
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|  Dharshan|
|   002|    102|Arun Kumar| 28|  Male|2023-06-15| 78000|         M|Arun Kumar|
|   003|    101|   Priya S| 24|Female|2024-03-10| 62000|         F|   Priya S|
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|  Dharshan|
|   004|    103|   Meena R|   |Female|2022-11-20|      |         F|   Meena R|
+------+-------+----------+---+------+----------+------+----------+----------+
only showing top 5 rows


In [13]:
# Date conversion: Convert hire_date from STRING to DATE type

from pyspark.sql.functions import to_date

emp_dated = emp_name_changed.withColumn(
    "hire_date", to_date(col("hire_date"), "yyyy-MM-dd")
)

In [14]:
# Check schema - hire_date is now DATE type
emp_dated.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- salary: string (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)


In [15]:
# Empty strings are now NULL after date conversion
emp_dated.show(10)

+------+-------+-------------+---+------+----------+------+----------+-------------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|new_gender|     new_name|
+------+-------+-------------+---+------+----------+------+----------+-------------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|         M|   Arun Kumar|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|         F|      Priya S|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|
|   004|    103|      Meena R|   |Female|2022-11-20|      |         F|      Meena R|
|   005|    102|    Saravanan| 35|  Male|      NULL| 95000|         M|    Saravanan|
|   006|    104|    Karthik P| 29|  Male|2024-09-05| 72000|         M|    Karthik P|
|   007|       |Deepika Menon| 26|Female|2023-02-28| 68000|         F|Deepika Menon|
|   008|    101|    Mohan Raj| 42|  Male|2020-05-12|120000|      

In [16]:
# Add current date and timestamp columns for tracking

from pyspark.sql.functions import current_date, current_timestamp

emp_with_dates = emp_dated.withColumn("date_now", current_date()).withColumn(
    "timestamp_now", current_timestamp()
)

In [17]:
# Current date and timestamp columns added
emp_with_dates.show(5)

+------+-------+----------+---+------+----------+------+----------+----------+----------+--------------------+
|emp_id|dept_id|      name|age|gender| hire_date|salary|new_gender|  new_name|  date_now|       timestamp_now|
+------+-------+----------+---+------+----------+------+----------+----------+----------+--------------------+
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|  Dharshan|2026-02-09|2026-02-09 11:37:...|
|   002|    102|Arun Kumar| 28|  Male|2023-06-15| 78000|         M|Arun Kumar|2026-02-09|2026-02-09 11:37:...|
|   003|    101|   Priya S| 24|Female|2024-03-10| 62000|         F|   Priya S|2026-02-09|2026-02-09 11:37:...|
|   001|    101|    Vishnu| 21|  Male|2025-01-01| 45000|         M|  Dharshan|2026-02-09|2026-02-09 11:37:...|
|   004|    103|   Meena R|   |Female|2022-11-20|      |         F|   Meena R|2026-02-09|2026-02-09 11:37:...|
+------+-------+----------+---+------+----------+------+----------+----------+----------+--------------------+
o

In [18]:
# Remove duplicate rows

emp_unique = emp_with_dates.dropDuplicates()

In [19]:
# Duplicates removed
emp_unique.show()

+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|new_gender|     new_name|  date_now|       timestamp_now|
+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|   009|    103|       Anjali| 23|     F|2024-07-19| 58000|      NULL|       Anjali|2026-02-09|2026-02-09 11:37:...|
|   010|    102| Ramesh Kumar| 31|     M|2021-10-01| 85000|      NULL| Ramesh Kumar|2026-02-09|2026-02-09 11:37:...|
|   013|    104|    Preethi K| 27|Female|      NULL| 64000|         F|    Preethi K|2026-02-09|2026-02-09 11:37:...|
|   016|       |       Sundar| 45|     M|2018-03-05| 92000|      NULL|       Sundar|2026-02-09|2026-02-09 11:37:...|
|   017|    101|    Kavya Sri| 22|     F|2025-03-01| 48000|      NULL|    Kavya Sri|2026-02-09|2026-02-09 11:37:...|
|   005|    102|    Saravanan| 35|  Male|      NULL| 95000|     

In [20]:
# Remove rows with any NULL values

emp_drop = emp_unique.dropna()

In [20]:
# Only rows without NULLs remain
emp_drop.show()

+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|new_gender|     new_name|  date_now|       timestamp_now|
+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|2026-02-09|2026-02-09 11:37:...|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|         M|   Arun Kumar|2026-02-09|2026-02-09 11:37:...|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|         F|      Priya S|2026-02-09|2026-02-09 11:37:...|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|2026-02-09|2026-02-09 11:37:...|
|   004|    103|      Meena R|   |Female|2022-11-20|      |         F|      Meena R|2026-02-09|2026-02-09 11:37:...|
|   006|    104|    Karthik P| 29|  Male|2024-09-05| 72000|     

In [21]:
# Final schema: hire_date is DATE type, date_now and timestamp_now added
emp_drop.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- salary: string (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- date_now: date (nullable = false)
 |-- timestamp_now: timestamp (nullable = false)


In [22]:
# Replace NULL values with default: Use coalesce to set 'O' for NULL gender

from pyspark.sql.functions import coalesce, lit

emp_null_df = emp_dated.withColumn("new_gender", coalesce(col("new_gender"), lit("O")))

In [23]:
# NULL gender values replaced with 'O' (see rows with M/F in original gender)
emp_null_df.show()

+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|new_gender|     new_name|  date_now|       timestamp_now|
+------+-------+-------------+---+------+----------+------+----------+-------------+----------+--------------------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|2026-02-09|2026-02-09 11:37:...|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|         M|   Arun Kumar|2026-02-09|2026-02-09 11:37:...|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|         F|      Priya S|2026-02-09|2026-02-09 11:37:...|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|         M|     Dharshan|2026-02-09|2026-02-09 11:37:...|
|   004|    103|      Meena R|   |Female|2022-11-20|      |         F|      Meena R|2026-02-09|2026-02-09 11:37:...|
|   005|    102|    Saravanan| 35|  Male|      NULL| 95000|     

In [ ]:
spark.stop()